<a href="https://colab.research.google.com/github/abaronia108/abaronia108/blob/main/MSDS_453_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
x_df = files.upload()

import pandas as pd
x_df = pd.read_csv("Mental-Health-Twitter 3.csv")

Saving Mental-Health-Twitter 3.csv to Mental-Health-Twitter 3.csv


In [ ]:
!pip install -U "gensim>=4.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 59.8 MB/s eta 0:00:00


In [ ]:
import gensim, numpy as np, scipy, sys
import os
import numpy as np
import re
import string
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
import random
from dataclasses import dataclass

from nltk.stem.wordnet import WordNetLemmatizer
from nltk.stem import PorterStemmer

from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

import scipy.cluster.hierarchy

from IPython.display import display, HTML

from typing import List, Callable, Dict

In [ ]:
"x_df.head()

,Unnamed: 0,post_id,post_created,post_text,user_id,followers,friends,favourites,statuses,retweets,label
0,0,637894677824413696,Sun Aug 30 07:48:37 +0000 2015,It's just over 2 years since I was diagnosed w...,1013187241,84,211,251,837,0,1
1,1,637890384576778240,Sun Aug 30 07:31:33 +0000 2015,"It's Sunday, I need a break, so I'm planning t...",1013187241,84,211,251,837,1,1
2,2,637749345908051968,Sat Aug 29 22:11:07 +0000 2015,Awake but tired. I need to sleep but my brain ...,1013187241,84,211,251,837,0,1
3,3,637696421077123073,Sat Aug 29 18:40:49 +0000 2015,RT @SewHQ: #Retro bears make perfect gifts and...,1013187241,84,211,251,837,2,1
4,4,637696327485366272,Sat Aug 29 18:40:26 +0000 2015,It’s hard to say whether packing lists are mak...,1013187241,84,211,251,837,1,1


In [ ]:
x_df.shape

(20000, 11)

In [ ]:
x_df["label"].value_counts()

,count
label,
1,10000
0,10000


In [ ]:
x_df = x_df[['post_text', 'label']]

In [ ]:
x_df.head()

,post_text,label
0,It's just over 2 years since I was diagnosed w...,1
1,"It's Sunday, I need a break, so I'm planning t...",1
2,Awake but tired. I need to sleep but my brain ...,1
3,RT @SewHQ: #Retro bears make perfect gifts and...,1
4,It’s hard to say whether packing lists are mak...,1


In [ ]:
!git clone https://github.com/Inusette/Identifying-depression

from sklearn.datasets import load_files

reddit_data = load_files('Identifying-depression/Data_Collector',
                         categories=['reddit_depression'], encoding='utf-8')

Cloning into 'Identifying-depression'...


In [ ]:
reddit_df = pd.DataFrame({
    'post_text': reddit_data.data,
    'label': reddit_data.target
})

In [ ]:
reddit_df.head()

,post_text,label
0,Does anyone else have this constant dark feeli...,0
1,Lost all motivation\nI recently have stopped d...,0
2,Mom always trying to ask me uncomfortable ques...,1
3,The days are manageable; it's the years that g...,0
4,Does anyone else get such intrusive thoughts l...,0


In [ ]:
x_df.shape

(20000, 2)

In [ ]:
reddit_df.shape

(1841, 2)

In [ ]:
print(reddit_data.target_names)

['reddit_depression', 'reddit_non_depression']


In [ ]:
reddit_df['label'] = 1 - reddit_df['label']

In [ ]:
reddit_df.head()

,post_text,label
0,Does anyone else have this constant dark feeli...,1
1,Lost all motivation\nI recently have stopped d...,1
2,Mom always trying to ask me uncomfortable ques...,0
3,The days are manageable; it's the years that g...,1
4,Does anyone else get such intrusive thoughts l...,1


In [ ]:
reddit_df['label'].value_counts()

,count
label,
1,1293
0,548


In [ ]:
x_df['total_words'] = x_df['post_text'].str.split().str.len()
reddit_df['total_words'] = reddit_df['post_text'].str.split().str.len()

In [ ]:
x_df.head()

,post_text,label,total_words
0,It's just over 2 years since I was diagnosed w...,1,26
1,"It's Sunday, I need a break, so I'm planning t...",1,19
2,Awake but tired. I need to sleep but my brain ...,1,13
3,RT @SewHQ: #Retro bears make perfect gifts and...,1,23
4,It’s hard to say whether packing lists are mak...,1,21


In [ ]:
reddit_df.head()

,post_text,label,total_words
0,Does anyone else have this constant dark feeli...,1,185
1,Lost all motivation\nI recently have stopped d...,1,45
2,Mom always trying to ask me uncomfortable ques...,0,340
3,The days are manageable; it's the years that g...,1,70
4,Does anyone else get such intrusive thoughts l...,1,100


In [ ]:
x_df['total_words'].median()

12.0

In [ ]:
reddit_df['total_words'].median()

139.0

# Preprocessing

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
# Normalize (remove punctuation, lower cases, removal of tags and special characters), tokenize, lemmatize, and remove the stop words
stop_words = set(nltk.corpus.stopwords.words('english'))
stop_words.update(['rt', 'RT'])
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'@\w+', " ", str(text))
    text = re.sub(r'#\w+', " ", str(text))
    text = re.sub(r'http\S+', " ", str(text))
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    tokenized_text = text.split()
    tokenized_text = [word for word in tokenized_text if word not in stop_words]
    lemmatized_text = [lemmatizer.lemmatize(x) for x in tokenized_text]
    return ' '.join(lemmatized_text)

x_df['clean_text'] = x_df['post_text'].apply(clean_text)
reddit_df['clean_text'] = reddit_df['post_text'].apply(clean_text)

In [ ]:
x_tokens = x_df['clean_text'].apply(lambda x: x.split())
reddit_tokens = reddit_df['clean_text'].apply(lambda x: x.split())

In [ ]:
x_df.head(50)

,post_text,label,clean_text
0,It's just over 2 years since I was diagnosed w...,1,year since diagnosed today taking moment refle...
1,"It's Sunday, I need a break, so I'm planning t...",1,sunday need break planning spend little time p...
2,Awake but tired. I need to sleep but my brain ...,1,awake tired need sleep brain idea
3,RT @SewHQ: #Retro bears make perfect gifts and...,1,bear make perfect gift great beginner get stit...
4,It’s hard to say whether packing lists are mak...,1,hard say whether packing list making life easi...
5,Making packing lists is my new hobby... #movin...,1,making packing list new hobby
6,At what point does keeping stuff for nostalgic...,1,point keeping stuff nostalgic reason cross lin...
7,Currently in the finding-boxes-of-random-shit ...,1,currently finding box random shit packing phas...
8,"Can't be bothered to cook, take away on the wa...",1,bothered cook take away way
9,RT @itventsnews: ITV releases promo video for ...,1,itv release promo video final series downton a...


In [ ]:
reddit_df.head(50)

,post_text,label,clean_text
0,Does anyone else have this constant dark feeli...,1,anyone else constant dark feeling lingering re...
1,Lost all motivation\nI recently have stopped d...,1,lost motivation recently stopped one thing enj...
2,Mom always trying to ask me uncomfortable ques...,0,mom always trying ask uncomfortable question g...
3,The days are manageable; it's the years that g...,1,day manageable year get day day aside struggle...
4,Does anyone else get such intrusive thoughts l...,1,anyone else get intrusive thought like hate cu...
5,"I'm tired, I need someone to tell me I'm okay....",1,tired need someone tell okay tell wrong feel m...
6,Advice to restore lost friendship?\nMy cousin ...,0,advice restore lost friendship cousin used bes...
7,Fear of leaving\nTl;dr; Sob story. I think i d...,0,fear leaving tl dr sob story think love mother...
8,Tomorrow is my 35th birthday.\n So tomorrow is...,1,tomorrow th birthday tomorrow th birthday real...
9,"What happened?\nHi guys,\n\nJust wondering if ...",0,happened hi guy wondering experienced grew hap...


# VADER Sentiment Analysis

In [ ]:
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.1 MB/s eta 0:00:00


In [ ]:
sentiment_score = SentimentIntensityAnalyzer()
def vader(text):
  return sentiment_score.polarity_scores(text)['compound']
x_df['sentiment_score'] = x_df['post_text'].apply(vader)
reddit_df['sentiment_score'] = reddit_df['post_text'].apply(vader)

#Vectorization

In [ ]:
tfidf = TfidfVectorizer()
X_tfidf_x = tfidf.fit_transform(x_df['clean_text'])
X_tfidf_reddit = tfidf.transform(reddit_df['clean_text'])

In [ ]:
tagged_x = [TaggedDocument(words=tokens, tags=[str(i)]) for i, tokens in enumerate(x_tokens)]
doc2vec_model = Doc2Vec(tagged_x, vector_size=100, min_count=2, epochs=30)
x_vectors = [doc2vec_model.infer_vector(tokens) for tokens in x_tokens]
reddit_vectors = [doc2vec_model.infer_vector(tokens) for tokens in reddit_tokens]

#Topic Modeling

###BERTopic

In [ ]:
!pip install bertopic

from bertopic import BERTopic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.2 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic

x_bertopic_depression = BERTopic()
x_bertopic_depression.fit_transform(x_df[x_df['label'] == 1]['post_text'].tolist())

x_bertopic_depression.get_topic_info()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3758,-1_and_the_you_to,"[and, the, you, to, rt, it, my, of, me, is]","[I'm so much in love~, @allthxngsozzie Even if..."
1,0,122,0_addiction_heroin_weed_addicted,"[addiction, heroin, weed, addicted, drug, drug...",[Do you know some who has been arrested becaus...
2,1,118,1_this_httptco4vpjikdqkl_httptco19if6s5pkm_web...,"[this, httptco4vpjikdqkl, httptco19if6s5pkm, w...","[what a website, bakabt is pretty nice, shame ..."
3,2,116,2_sleep_tired_night_awake,"[sleep, tired, night, awake, bed, goodnight, a...","[I just want to sleep., I NEED SLEEP, im need ..."
4,3,111,3_overcome_depression_circles_therapy,"[overcome, depression, circles, therapy, end, ...",[Overcome Depression: KUWTK: Khloe Kardashian ...
...,...,...,...,...,...
173,172,10,172_location_0229_derpcon_nue,"[location, 0229, derpcon, nue, annandale, belf...",[@_5SOSFAMUPDATES omg does that mean that derp...
174,173,10,173_gratitude_powerful_httpstcohmteuotku3_http...,"[gratitude, powerful, httpstcohmteuotku3, http...",[Having #gratitude has definitely improved our...
175,174,10,174_bed_house_doodlethebug_her,"[bed, house, doodlethebug, her, lady, side, ca...",[Time for a new bed since mine just decided to...
176,175,10,175_gentlemans_kidtheatre_murder_guide,"[gentlemans, kidtheatre, murder, guide, movie,...",[RT @broadwayworld: A GENTLEMAN'S GUIDE TO LOV...


In [ ]:
topic_info = x_bertopic_depression.get_topic_info()
topic_info[["Topic", "Count", "Name"]].sort_values("Count", ascending=False).head(15)

,Topic,Count,Name
0,-1,3758,-1_and_the_you_to
1,0,122,0_addiction_heroin_weed_addicted
2,1,118,1_this_httptco4vpjikdqkl_httptco19if6s5pkm_web...
3,2,116,2_sleep_tired_night_awake
4,3,111,3_overcome_depression_circles_therapy
5,4,108,4_mental_health_mentalhealth_illness
6,5,103,5_album_music_song_good
7,6,99,6_hair_wear_dress_color
8,7,96,7_allthxngsozzie_yes_slay_glue
9,8,93,8_lizsmith333_tauri3l_thefuxedos_genevieveverso


In [ ]:
x_bertopic_depression.visualize_topics(topics=[0, 2, 3, 4, 9,12, 13])


In [ ]:
x_bertopic_depression.visualize_barchart(topics=[0, 2, 3, 4, 9,12, 13])

In [ ]:
vectorizer_model = CountVectorizer(max_df=.8)
reddit_bertopic_depression = BERTopic(vectorizer_model=vectorizer_model)
reddit_bertopic_depression.fit_transform(reddit_df[reddit_df['label'] == 1]['post_text'].tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

([-1,
  7,
  0,
  -1,
  2,
  9,
  -1,
  -1,
  -1,
  -1,
  -1,
  -1,
  4,
  -1,
  -1,
  -1,
  -1,
  20,
  -1,
  7,
  -1,
  -1,
  4,
  -1,
  0,
  9,
  -1,
  4,
  2,
  2,
  0,
  7,
  6,
  -1,
  18,
  -1,
  12,
  3,
  13,
  2,
  -1,
  -1,
  0,
  5,
  -1,
  -1,
  -1,
  15,
  -1,
  4,
  10,
  -1,
  10,
  -1,
  -1,
  5,
  9,
  17,
  16,
  2,
  13,
  -1,
  -1,
  2,
  -1,
  7,
  0,
  3,
  -1,
  5,
  3,
  -1,
  -1,
  13,
  14,
  4,
  15,
  6,
  -1,
  -1,
  -1,
  13,
  10,
  2,
  6,
  11,
  5,
  3,
  6,
  -1,
  -1,
  3,
  -1,
  -1,
  4,
  -1,
  -1,
  18,
  -1,
  -1,
  2,
  2,
  0,
  -1,
  0,
  18,
  -1,
  0,
  -1,
  9,
  2,
  10,
  8,
  10,
  -1,
  -1,
  -1,
  3,
  -1,
  -1,
  -1,
  4,
  7,
  5,
  3,
  4,
  -1,
  5,
  3,
  0,
  -1,
  12,
  11,
  0,
  5,
  8,
  9,
  6,
  3,
  16,
  7,
  5,
  6,
  -1,
  1,
  14,
  -1,
  -1,
  8,
  -1,
  1,
  3,
  8,
  15,
  15,
  -1,
  -1,
  -1,
  12,
  -1,
  3,
  4,
  -1,
  1,
  -1,
  5,
  8,
  -1,
  18,
  -1,
  0,
  -1,
  -1,
  14,
  -1,
  -1,
  -1,
  3,
  12,
  

In [ ]:
topic_info = reddit_bertopic_depression.get_topic_info()
topic_info[["Topic", "Count", "Name"]].sort_values("Count", ascending=False).head(15)

,Topic,Count,Name
0,-1,558,-1_we_he_were_job
1,0,98,0_die_care_exist_alive
2,1,72,1_alone_hang_lonely_friend
3,2,70,2_school_job_college_high
4,3,70,3_she_her_we_girl
5,4,58,4_cry_sad_crying_sadness
6,5,42,5_sleep_bed_wake_nightmares
7,6,39,6_she_her_mom_shes
8,7,31,7_motivation_lazy_tired_job
9,8,29,8_anxiety_meds_illness_mental


In [ ]:
reddit_bertopic_depression.visualize_barchart(topics=[11, 10, 8, 7, 5, 4, 2, 1, 0])

In [ ]:
reddit_bertopic_depression.visualize_topics(topics=[11, 10, 8, 7, 5, 4, 2, 1, 0])

### LDA

In [ ]:
!pip install mglearn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 38.6 MB/s eta 0:00:00


In [ ]:
import mglearn


In [ ]:
# LDA
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

stop_words = ['like', 'much', 'would', 'get', 'may', 'gt', 'one' , 'amp', 'get', 'got', 'via', 'make', 'new' 'really', 'right', 'want', 'im']

count_vectorizer = CountVectorizer(max_features = 10000, max_df = .6, min_df = 2, stop_words = stop_words)
vectorized_text = count_vectorizer.fit_transform(x_df[x_df['label'] == 1]['clean_text'])


In [ ]:
lda_test = LatentDirichletAllocation(n_components=5, random_state=42)
lda_test.fit(vectorized_text)
print('perplexity:',lda_test.perplexity(vectorized_text))

lda_test = LatentDirichletAllocation(n_components=10, random_state=42)
lda_test.fit(vectorized_text)
print('perplexity:',lda_test.perplexity(vectorized_text))

perplexity: 2834.936451519838
perplexity: 3240.4392790367665


In [ ]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)
topics = lda.fit_transform(vectorized_text)

feature_names = np.array(count_vectorizer.get_feature_names_out())
sorted_indices = np.argsort(lda.components_)[:, ::-1]

mglearn.tools.print_topics(topics=range(5), feature_names=feature_names,
sorting=sorted_indices, topics_per_chunk=5, n_words=10)

topic 0       topic 1       topic 2       topic 3       topic 4       
--------      --------      --------      --------      --------      
love          depression    depression    time          go            
know          talk          treatment     day           please        
people        see           help          good          guy           
thing         positive      headache      never         great         
shit          life          overcome      hate          day           
feel          today         health        work          god           
fuck          overcome      migraine      happy         morning       
ever          treatment     disorder      sleep         time          
lt            know          therapy       girl          thank         
good          someone       end           year          new           




In [ ]:
count_vectorizer = CountVectorizer(max_features = 10000, max_df = .8, min_df = 2, stop_words = stop_words)
vectorized_text = count_vectorizer.fit_transform(reddit_df[reddit_df['label'] == 1]['clean_text'])


In [ ]:
lda_test = LatentDirichletAllocation(n_components=5, random_state=42)
lda_test.fit(vectorized_text)
print('perplexity:',lda_test.perplexity(vectorized_text))

lda_test = LatentDirichletAllocation(n_components=10, random_state=42)
lda_test.fit(vectorized_text)
print('perplexity:',lda_test.perplexity(vectorized_text))

perplexity: 1388.2448753455894
perplexity: 1533.5731357936618


In [ ]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)
topics = lda.fit_transform(vectorized_text)

feature_names = np.array(count_vectorizer.get_feature_names_out())
sorted_indices = np.argsort(lda.components_)[:, ::-1]

mglearn.tools.print_topics(topics=range(5), feature_names=feature_names,
sorting=sorted_indices, topics_per_chunk=5, n_words=10)

topic 0       topic 1       topic 2       topic 3       topic 4       
--------      --------      --------      --------      --------      
feel          feel          year          know          stranger      
life          depression    time          feel          depression    
friend        people        feel          life          feel          
people        friend        life          people        help          
know          know          back          time          thing         
really        really        day           thing         need          
even          time          week          even          know          
think         even          know          day           life          
thing         day           friend        depression    better        
anything      love          month         go            people        




# DistilBERT Transfomer Sentiment Analysis

In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
classified_result = classifier(
    x_df['post_text'].tolist(),
    truncation = True,
    padding = True,
    max_length = 512
)

reddit_classified_result = classifier(
    reddit_df['post_text'].tolist(),
    truncation = True,
    padding = True,
    max_length = 512
)

In [ ]:
x_df['transformer_sentiment'] = list(map(lambda f: f['label'], classified_result))
reddit_df['transformer_sentiment'] = list(map(lambda f: f['label'], reddit_classified_result))

# Classification

In [ ]:
from sklearn.model_selection import train_test_split
x = x_df['clean_text']
y = x_df['label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Logistic Regression Classification

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
# Classification using TF-IDF for X

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

X_tfidf_train = tfidf.fit_transform(x_train)
X_tfidf_test = tfidf.transform(x_test)

logistic_regressor = LogisticRegression()
logistic_regressor.fit(X_tfidf_train, y_train)
print("X TF-IDF Accuracy",logistic_regressor.score(X_tfidf_test, y_test))

y_pred_test = logistic_regressor.predict(X_tfidf_test)
print("F1 score:",f1_score(y_test, y_pred_test))


X TF-IDF Accuracy 0.7475
F1 score: 0.754735308402137


In [ ]:
# Classification using TF-IDF for Reddit

from sklearn.metrics import accuracy_score

X_tfidf_reddit = tfidf.transform(reddit_df['clean_text'])
y_pred_reddit = logistic_regressor.predict(X_tfidf_reddit)
print('Reddit TF-IDF Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))

print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit TF-IDF Accuracy: 0.7007061379684953
F1 score: 0.8188096021045709


In [ ]:
# Classification using Doc2Vec for X
X_doc2vec_train = np.array(x_vectors)[x_train.index]
X_doc2vec_test = np.array(x_vectors)[x_test.index]
logistic_regressor.fit(X_doc2vec_train, y_train)
print("X Doc2Vec Accuracy:",logistic_regressor.score(X_doc2vec_test, y_test))
x = logistic_regressor.predict(X_doc2vec_test)
print("F1 score:",f1_score(y_test, x))


X Doc2Vec Accuracy: 0.6305
F1 score: 0.6334325396825397


In [ ]:
# Classification using Dodc2Vec for Reddit
X_doc2vec_reddit = np.array(reddit_vectors)
y_pred_reddit = logistic_regressor.predict(X_doc2vec_reddit)
print('Reddit Doc2Vec Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit Doc2Vec Accuracy: 0.7034220532319392
F1 score: 0.8171466845277964


In [ ]:
x_df['transformer_sentiment_prediction'] = np.where(x_df['transformer_sentiment'] == 'POSITIVE', 1, -1)

In [ ]:
# Classification using DistilBERT for X

X_transformer_train = x_df['transformer_sentiment_prediction'].iloc[x_train.index]
X_transformer_test = x_df['transformer_sentiment_prediction'].iloc[x_test.index]

logistic_regressor.fit(X_transformer_train.to_frame(), y_train)
print("X DistilBERT Accuracy:",logistic_regressor.score(X_transformer_test.to_frame(), y_test))
print("F1 score:",f1_score(y_test, logistic_regressor.predict(X_transformer_test.to_frame())))

X DistilBERT Accuracy: 0.512
F1 score: 0.5958592132505176


In [ ]:
# Classification using DistilBERT for Reddit

reddit_df['transformer_sentiment_prediction'] = np.where(reddit_df['transformer_sentiment'] == 'POSITIVE', 1, -1)
X_transformer_reddit = reddit_df['transformer_sentiment_prediction']
y_pred_reddit = logistic_regressor.predict(X_transformer_reddit.to_frame())
print('Reddit DistilBERT Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit DistilBERT Accuracy: 0.6556219445953286
F1 score: 0.7784765897973445


In [ ]:
# Correlations b/w DistilBERT and VADER

print("Correlation between DistilBERT and VADER sentiment scores for X", x_df['sentiment_score'].corr(x_df['transformer_sentiment_prediction']))
print("Correlation between DistilBERT and VADER sentiment scores for Reddit",reddit_df['sentiment_score'].corr(reddit_df['transformer_sentiment_prediction']))

Correlation between DistilBERT and VADER sentiment scores for X 0.4324722616799021
Correlation between DistilBERT and VADER sentiment scores for Reddit 0.28018692569463516


In [ ]:
# Classification using VADER sentiment score for X
X_vader_train = x_df['sentiment_score'].iloc[x_train.index]
X_vader_test = x_df['sentiment_score'].iloc[x_test.index]
logistic_regressor.fit(X_vader_train.to_frame(), y_train)
print("X VADER Accuracy:",logistic_regressor.score(X_vader_test.to_frame(), y_test))
print("F1 score:",f1_score(y_test, logistic_regressor.predict(X_vader_test.to_frame())))

X VADER Accuracy: 0.48175
F1 score: 0.5338430402518551


In [ ]:
# Classification using VADER sentiment score for Reddit
X_vader_reddit = reddit_df['sentiment_score']
y_pred_reddit = logistic_regressor.predict(X_vader_reddit.to_frame())
print('Reddit VADER Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit VADER Accuracy: 0.6480173818576861
F1 score: 0.7268128161888702


### Naive Bayes Classification

In [ ]:
# Classification using TF-IDF for X using Naive Bayes

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

naive_bayes = MultinomialNB()
naive_bayes.fit(X_tfidf_train, y_train)
print("X TF-IDF Accuracy",naive_bayes.score(X_tfidf_test, y_test))
print("F1 score:",f1_score(y_test, naive_bayes.predict(X_tfidf_test)))

X TF-IDF Accuracy 0.76
F1 score: 0.7740112994350282


In [ ]:
# Classification using TF-IDF for Reddit

y_pred_reddit = naive_bayes.predict(X_tfidf_reddit)
print('Reddit TF-IDF Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))


Reddit TF-IDF Accuracy: 0.7028788701792504
F1 score: 0.821998047510576


In [ ]:
# Classification using Doc2Vec for X

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

gaussian_naive_bayes = GaussianNB()

gaussian_naive_bayes.fit(X_doc2vec_train, y_train)
print("X Doc2Vec Accuracy:",gaussian_naive_bayes.score(X_doc2vec_test, y_test))
print("F1 score:",f1_score(y_test, gaussian_naive_bayes.predict(X_doc2vec_test)))


X Doc2Vec Accuracy: 0.5825
F1 score: 0.5368829728230726


In [ ]:
# Classification using Doc2Vec for Reddit

y_pred_reddit = gaussian_naive_bayes.predict(X_doc2vec_reddit)
print('Reddit Doc2Vec Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit Doc2Vec Accuracy: 0.7034220532319392
F1 score: 0.8238709677419355


In [ ]:
# Classification using DistilBERT for X

gaussian_naive_bayes.fit(X_transformer_train.to_frame(), y_train)
print("X DistilBERT Accuracy:",gaussian_naive_bayes.score(X_transformer_test.to_frame(), y_test))
print("F1 score:",f1_score(y_test, gaussian_naive_bayes.predict(X_transformer_test.to_frame())))

X DistilBERT Accuracy: 0.512
F1 score: 0.5958592132505176


In [ ]:
# Classification using DistilBERT for Reddit

y_pred_reddit = gaussian_naive_bayes.predict(X_transformer_reddit.to_frame())
print('Reddit DistilBERT Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit DistilBERT Accuracy: 0.6556219445953286
F1 score: 0.7784765897973445


In [ ]:
# Classification using VADER sentiment score for X

gaussian_naive_bayes.fit(X_vader_train.to_frame(), y_train)
print("X VADER Accuracy:",gaussian_naive_bayes.score(X_vader_test.to_frame(), y_test))
print("F1 score:",f1_score(y_test, gaussian_naive_bayes.predict(X_vader_test.to_frame())))

X VADER Accuracy: 0.53475
F1 score: 0.4348618281202551


In [ ]:
# Classification using VADER sentiment score for Reddit

y_pred_reddit = gaussian_naive_bayes.predict(X_vader_reddit.to_frame())
print('Reddit VADER Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit VADER Accuracy: 0.6474741988049972
F1 score: 0.7681314755269739


### Random Forest Classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
# Classification using TF-IDF for X with RandomForest
random_forest = RandomForestClassifier()

random_forest.fit(X_tfidf_train, y_train)
print("X TF-IDF Accuracy",random_forest.score(X_tfidf_test, y_test))
print("F1 score:",f1_score(y_test, random_forest.predict(X_tfidf_test)))

X TF-IDF Accuracy 0.73075
F1 score: 0.7501739735560194


In [ ]:
# Classification using TF-IDF for Reddit

y_pred_reddit = random_forest.predict(X_tfidf_reddit)
print('Reddit TF-IDF Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))


Reddit TF-IDF Accuracy: 0.6827810972297664
F1 score: 0.7863935625457206


In [ ]:
# Classification using Doc2Vec for X

random_forest.fit(X_doc2vec_train, y_train)
print("X Doc2Vec Accuracy:",random_forest.score(X_doc2vec_test, y_test))
print("F1 score:",f1_score(y_test, random_forest.predict(X_doc2vec_test)))

X Doc2Vec Accuracy: 0.66975
F1 score: 0.6808407827977773


In [ ]:
# Classification using Doc2Vec for Reddit

y_pred_reddit = random_forest.predict(X_doc2vec_reddit)
print('Reddit Doc2Vec Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit Doc2Vec Accuracy: 0.5437262357414449
F1 score: 0.6146788990825688


In [ ]:
# Classification using DistilBERT for X

random_forest.fit(X_transformer_train.to_frame(), y_train)
print("X DistilBERT Accuracy:",random_forest.score(X_transformer_test.to_frame(), y_test))
print("F1 score:", f1_score(y_test, random_forest.predict(X_transformer_test.to_frame())))

X DistilBERT Accuracy: 0.512
F1 score: 0.5958592132505176


In [ ]:
# Classification using DistilBERT for Reddit

y_pred_reddit = random_forest.predict(X_transformer_reddit.to_frame())
print('Reddit DistilBERT Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))

Reddit DistilBERT Accuracy: 0.6556219445953286
F1 score: 0.7784765897973445


In [ ]:
# Classification using VADER sentiment score for X

random_forest.fit(X_vader_train.to_frame(), y_train)
print("X VADER Accuracy:",random_forest.score(X_vader_test.to_frame(), y_test))
print("F1 score:", f1_score(y_test, random_forest.predict(X_vader_test.to_frame())))

X VADER Accuracy: 0.57725
F1 score: 0.5558182295770948


In [ ]:
# Classification using VADER sentiment score for Reddit

y_pred_reddit = random_forest.predict(X_vader_reddit.to_frame())
print('Reddit VADER Accuracy:' , accuracy_score(reddit_df['label'], y_pred_reddit))
print("F1 score:",f1_score(reddit_df['label'], y_pred_reddit))


Reddit VADER Accuracy: 0.5285171102661597
F1 score: 0.6162687886825818
